# CRSP market-feature audit

This notebook audits the point-in-time market features produced by `scripts/12_build_crsp_market_features.py`.

It checks:

1. the integrity and coverage of the stock-quarter feature table;
2. whether all feature windows precede the information date;
3. reconciliation with the saved quarterly audit; and
4. feature usability, missingness, and distributions.

The notebook reads the saved outputs only and does not rebuild the features.

## 1. Setup and inputs

In [1]:
import sys
from pathlib import Path

import duckdb
import pandas as pd
import pyarrow.parquet as pq


project_root = Path.cwd().resolve()

while not (project_root / "config" / "paths.yaml").is_file():
    if project_root.parent == project_root:
        raise FileNotFoundError("Could not locate the project root.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.configuration import get_paths_config


paths = get_paths_config()
output_directory = paths.interim / "crsp"
features_path = output_directory / "market_features.parquet"
audit_path = output_directory / "market_feature_audit.parquet"
security_nodes_path = (
    paths.interim / "networks" / "ownership" / "security_nodes.parquet"
)

table_paths = {
    "market_features": features_path,
    "market_feature_audit": audit_path,
    "security_nodes": security_nodes_path,
}

for name, path in table_paths.items():
    if not path.is_file():
        raise FileNotFoundError(f"Required {name} file not found: {path}")

inventory = pd.DataFrame(
    {
        "table": table_paths.keys(),
        "rows": [
            pq.ParquetFile(path).metadata.num_rows
            for path in table_paths.values()
        ],
    }
)

connection = duckdb.connect(database=":memory:")
connection.read_parquet(str(features_path)).create_view("market_features")
connection.read_parquet(str(audit_path)).create_view("saved_audit")
connection.read_parquet(str(security_nodes_path)).create_view("security_nodes")

display(inventory.style.format({"rows": "{:,}"}))

,table,rows
0,market_features,"206,868"
1,market_feature_audit,52
2,security_nodes,"206,868"


## 2. Feature-table integrity

Each ownership-network security node should have exactly one feature row. Historical windows must end strictly before the information date, and the saved usability flag must follow the fixed observation-count and market-data requirements.

In [2]:
check_values = connection.execute(
    '''
    WITH feature_checks AS (
        SELECT
            COUNT(*) - COUNT(DISTINCT (report_period, permno))
                AS duplicate_feature_keys,
            COUNT(*) FILTER (
                WHERE report_period IS NULL
                   OR information_date IS NULL
                   OR permno IS NULL
            ) AS missing_feature_keys,
            COUNT(*) FILTER (
                WHERE TRY_STRPTIME(PERIODOFREPORT, '%d-%b-%Y') IS NULL
                   OR TRY_STRPTIME(PERIODOFREPORT, '%d-%b-%Y')::DATE
                        != CAST(report_period AS DATE)
            ) AS inconsistent_report_periods,
            COUNT(*) FILTER (
                WHERE feature_window_start IS NULL
                   OR feature_window_end IS NULL
                   OR feature_window_start > feature_window_end
                   OR feature_window_end >= information_date
            ) AS invalid_feature_windows,
            COUNT(*) FILTER (
                WHERE return_observations_21d NOT BETWEEN 0 AND 21
                   OR return_observations_63d NOT BETWEEN 0 AND 63
                   OR return_observations_126d NOT BETWEEN 0 AND 126
                   OR return_observations_252d NOT BETWEEN 0 AND 252
                   OR beta_return_observations_252d NOT BETWEEN 0 AND 252
                   OR return_observations_21d > return_observations_63d
                   OR return_observations_63d > return_observations_126d
                   OR return_observations_126d > return_observations_252d
                   OR beta_return_observations_252d > return_observations_252d
            ) AS invalid_observation_counts,
            COUNT(*) FILTER (
                WHERE (volatility_63d IS NOT NULL AND volatility_63d < 0)
                   OR (volatility_252d IS NOT NULL AND volatility_252d < 0)
                   OR (downside_volatility_63d IS NOT NULL
                       AND downside_volatility_63d < 0)
                   OR (downside_volatility_252d IS NOT NULL
                       AND downside_volatility_252d < 0)
                   OR (negative_return_share_63d IS NOT NULL
                       AND NOT negative_return_share_63d BETWEEN 0 AND 1)
                   OR (maximum_drawdown_252d IS NOT NULL
                       AND NOT maximum_drawdown_252d BETWEEN 0 AND 1)
            ) AS invalid_feature_values,
            COUNT(*) FILTER (
                WHERE market_feature_usable IS DISTINCT FROM (
                    feature_window_start IS NOT NULL
                    AND return_observations_63d >= 50
                    AND return_observations_252d >= 202
                    AND beta_return_observations_252d >= 202
                    AND market_beta_252d IS NOT NULL
                    AND market_price_usd > 0
                    AND market_cap_usd > 0
                )
            ) AS incorrect_usability_flags
        FROM market_features
    ),
    feature_keys AS (
        SELECT report_period, permno FROM market_features
    ),
    node_keys AS (
        SELECT report_period, permno FROM security_nodes
    ),
    coverage_checks AS (
        SELECT
            COUNT(*) FILTER (WHERE f.permno IS NULL) AS nodes_without_features,
            COUNT(*) FILTER (WHERE n.permno IS NULL) AS features_without_nodes
        FROM node_keys n
        FULL JOIN feature_keys f USING (report_period, permno)
    )
    SELECT * FROM feature_checks CROSS JOIN coverage_checks
    '''
).fetchdf().iloc[0]

integrity_checks = pd.DataFrame(
    {
        "check": check_values.index.str.replace("_", " "),
        "violations": check_values.values,
    }
)

display(integrity_checks.style.format({"violations": "{:,}"}))
print()
print(f"Total feature-table violations: {integrity_checks['violations'].sum():,}")

,check,violations
0,duplicate feature keys,0
1,missing feature keys,0
2,inconsistent report periods,0
3,invalid feature windows,0
4,invalid observation counts,0
5,invalid feature values,0
6,incorrect usability flags,0
7,nodes without features,0
8,features without nodes,0



Total feature-table violations: 0


## 3. Quarterly-audit reconciliation

The quarterly audit is independently reproduced from the feature table. Counts, usability, observation coverage, and missingness should agree exactly apart from negligible floating-point error in the usability share.

In [3]:
audit_reconciliation = connection.execute(
    '''
    WITH calculated AS (
        SELECT
            PERIODOFREPORT,
            report_period,
            information_date,
            COUNT(*) AS security_periods,
            COUNT(*) FILTER (WHERE market_feature_usable)
                AS usable_security_periods,
            AVG(CAST(market_feature_usable AS DOUBLE)) AS usable_share,
            MEDIAN(return_observations_252d)
                AS median_return_observations_252d,
            COUNT(*) FILTER (WHERE market_cap_usd IS NULL)
                AS missing_market_cap,
            COUNT(*) FILTER (WHERE market_beta_252d IS NULL)
                AS missing_market_beta,
            COUNT(*) FILTER (
                WHERE return_252d IS NULL
                   OR volatility_252d IS NULL
                   OR market_beta_252d IS NULL
                   OR maximum_drawdown_252d IS NULL
            ) AS missing_primary_features
        FROM market_features
        GROUP BY PERIODOFREPORT, report_period, information_date
    )
    SELECT
        (
            SELECT COUNT(*) - COUNT(DISTINCT report_period)
            FROM saved_audit
        ) AS duplicate_audit_quarters,
        COUNT(*) FILTER (WHERE c.report_period IS NULL)
            AS audits_without_feature_rows,
        COUNT(*) FILTER (WHERE a.report_period IS NULL)
            AS feature_quarters_without_audits,
        COUNT(*) FILTER (
            WHERE a.PERIODOFREPORT != c.PERIODOFREPORT
               OR a.information_date != c.information_date
               OR a.security_periods != c.security_periods
               OR a.usable_security_periods != c.usable_security_periods
               OR abs(a.usable_share - c.usable_share) > 1e-12
               OR a.median_return_observations_252d
                    != c.median_return_observations_252d
               OR a.missing_market_cap != c.missing_market_cap
               OR a.missing_market_beta != c.missing_market_beta
               OR a.missing_primary_features != c.missing_primary_features
        ) AS quarterly_statistic_mismatches
    FROM saved_audit a
    FULL JOIN calculated c USING (report_period)
    '''
).fetchdf().T.reset_index()
audit_reconciliation.columns = ["check", "violations"]

display(audit_reconciliation.style.format({"violations": "{:,}"}))
print()
print(
    "Total quarterly-reconciliation violations: "
    f"{audit_reconciliation['violations'].sum():,}"
)

,check,violations
0,duplicate_audit_quarters,0
1,audits_without_feature_rows,0
2,feature_quarters_without_audits,0
3,quarterly_statistic_mismatches,0



Total quarterly-reconciliation violations: 0


## 4. Feature usability

A stock-quarter is usable when it has sufficient 63- and 252-day return history, sufficient observations for beta, a defined beta, and positive price and market capitalization. Reason counts can overlap because one observation may fail more than one requirement.

In [4]:
usability_summary = connection.execute(
    '''
    SELECT
        COUNT(*) AS stock_quarters,
        COUNT(*) FILTER (WHERE market_feature_usable) AS usable_stock_quarters,
        AVG(CAST(market_feature_usable AS DOUBLE)) AS usable_share,
        COUNT(*) FILTER (WHERE return_observations_63d < 50)
            AS insufficient_63d_returns,
        COUNT(*) FILTER (WHERE return_observations_252d < 202)
            AS insufficient_252d_returns,
        COUNT(*) FILTER (WHERE beta_return_observations_252d < 202)
            AS insufficient_beta_returns,
        COUNT(*) FILTER (WHERE market_beta_252d IS NULL) AS missing_beta,
        COUNT(*) FILTER (WHERE market_price_usd <= 0 OR market_price_usd IS NULL)
            AS invalid_market_price,
        COUNT(*) FILTER (WHERE market_cap_usd <= 0 OR market_cap_usd IS NULL)
            AS invalid_market_cap
    FROM market_features
    '''
).fetchdf()
overall_usability = usability_summary[[
    "stock_quarters",
    "usable_stock_quarters",
    "usable_share",
]]
failure_counts = (
    usability_summary.drop(columns=overall_usability.columns)
    .T.reset_index()
)
failure_counts.columns = ["failed_requirement", "stock_quarters"]

quarterly_usability = connection.execute(
    '''
    SELECT report_period, security_periods, usable_security_periods, usable_share
    FROM saved_audit
    ORDER BY report_period
    '''
).fetchdf()
quarterly_range = quarterly_usability[["usable_share"]].agg(
    ["min", "median", "max"]
).T

display(overall_usability.style.format({
    "stock_quarters": "{:,}",
    "usable_stock_quarters": "{:,}",
    "usable_share": "{:.2%}",
}))
display(failure_counts.style.format({"stock_quarters": "{:,}"}))
display(quarterly_range.style.format("{:.2%}"))
display(
    quarterly_usability.nsmallest(5, "usable_share").style.format({
        "security_periods": "{:,}",
        "usable_security_periods": "{:,}",
        "usable_share": "{:.2%}",
    })
)

,stock_quarters,usable_stock_quarters,usable_share
0,"206,868","196,681",95.08%


,failed_requirement,stock_quarters
0,insufficient_63d_returns,"2,014"
1,insufficient_252d_returns,"9,205"
2,insufficient_beta_returns,"9,205"
3,missing_beta,0
4,invalid_market_price,0
5,invalid_market_cap,0


,min,median,max
usable_share,86.85%,95.84%,97.81%


,report_period,security_periods,usable_security_periods,usable_share
33,2021-09-30 00:00:00,"4,419","3,838",86.85%
32,2021-06-30 00:00:00,"4,279","3,726",87.08%
31,2021-03-31 00:00:00,"4,103","3,642",88.76%
34,2021-12-31 00:00:00,"4,547","4,051",89.09%
35,2022-03-31 00:00:00,"4,598","4,186",91.04%


## 5. Usable-feature distributions

The table describes the principal market controls among usable stock-quarters. Extreme values are retained here for transparent diagnosis; any modeling transformation is defined later and must be fitted on training data only.

In [5]:
feature_columns = [
    "market_cap_usd",
    "average_daily_dollar_volume_63d",
    "return_252d",
    "momentum_252_21d",
    "volatility_252d",
    "downside_volatility_252d",
    "market_beta_252d",
    "maximum_drawdown_252d",
]

feature_summaries = []

for column in feature_columns:
    summary = connection.execute(
        f'''
        SELECT
            COUNT({column}) AS observations,
            QUANTILE_CONT({column}, 0.01) AS p01,
            MEDIAN({column}) AS median,
            QUANTILE_CONT({column}, 0.99) AS p99
        FROM market_features
        WHERE market_feature_usable
        '''
    ).fetchdf().iloc[0].to_dict()
    feature_summaries.append({"feature": column, **summary})

feature_distribution = pd.DataFrame(feature_summaries)

display(feature_distribution.style.format({
    "observations": "{:,}",
    "p01": "{:,.3f}",
    "median": "{:,.3f}",
    "p99": "{:,.3f}",
}))

connection.close()

,feature,observations,p01,median,p99
0,market_cap_usd,"196,681.0","4,897,142.000","765,368,240.000","161,227,775,894.001"
1,average_daily_dollar_volume_63d,"196,681.0","12,809.441","6,239,215.049","934,629,278.793"
2,return_252d,"196,681.0",-0.922,0.033,2.738
3,momentum_252_21d,"196,681.0",-0.902,0.028,2.544
4,volatility_252d,"196,681.0",0.112,0.423,2.244
5,downside_volatility_252d,"196,681.0",0.073,0.286,1.026
6,market_beta_252d,"196,681.0",-0.184,1.026,2.752
7,maximum_drawdown_252d,"196,681.0",0.054,0.340,0.952


## 6. Interpretation

- **The feature table has complete one-to-one coverage.** It contains 206,868 unique stock-quarter rows, exactly matching the ownership-network security nodes. All keys, report periods, windows, observation counts, feature domains, and usability flags pass their checks.
- **The construction is point in time.** Every historical feature window ends strictly before its quarter-specific information date, so the saved market controls do not use contemporaneous or future returns.
- **The quarterly audit is reproducible.** All 52 quarters are unique, and the saved counts, usability shares, observation coverage, and missingness agree exactly with an independent aggregation of the feature table.
- **Most stock-quarters are usable.** The fixed rule retains 196,681 observations, or 95.08% of the feature table. Quarterly usability ranges from 86.85% to 97.81%, with a median of 95.84%.
- **Insufficient trading history drives exclusion.** There are 9,205 stock-quarters with fewer than 202 observations for the 252-day return and beta windows, and 2,014 with fewer than 50 observations in the 63-day window. These reason counts overlap. No observation fails because of a missing estimated beta or a nonpositive final price or market capitalization.
- **The usable features retain meaningful cross-sectional variation.** The median stock-quarter has market beta 1.026, annualized 252-day volatility 0.423, downside volatility 0.286, and maximum drawdown 0.340. The wide return, size, liquidity, and risk distributions should be handled by the predefined modeling transformations rather than altered in this audit.

The Script 12 outputs are internally consistent and suitable for construction of the stock-quarter modeling panel.